# Robot arm example

In [ ]:
#@title Install libraries

!pip -q install vedo

In [ ]:
#@title Import libraries
import vedo

if "google.colab" in str(get_ipython()):
    vedo.settings.init_colab()


import os
import glob
import numpy as np
import vedo

# Force VTK backend (important in Colab)
if "google.colab" in str(get_ipython()):
    if vedo.settings.default_backend != "vtk":
        print("Switching VTK backend for compatibility.")
        vedo.settings.default_backend = "vtk"



In [ ]:
# Download zip file containing robot parts from dropbox
# Only if folder robots/ does not exist
if not os.path.exists("robot/"):
  !wget -O robot.zip "https://www.dropbox.com/scl/fi/uewvrcempf2wf2jp7bcb8/robot.zip?rlkey=7uwz1ne94hxyinub8x16y93em&dl=1"
  !unzip robot.zip
  !rm robot.zip

## RobotArm class

In [ ]:
# from vedo import LinearTransform


# class RobotArm():
# 	'''
# 		Robot arm.
# 	'''

# 	def __init__(self, partLengths, parts, arm_location):
# 		'''
# 			Initialize object
# 		'''

# 		# Arm location (position of the first frame)
# 		self.arm_location = arm_location

# 		# Length of the parts
# 		self.L1, self.L2, self.L3, self.L4 = partLengths

# 		# Constants
# 		self.delta_phi = 0.1
# 		self.target = [0, 100, 200]
# 		self.target_tolerance = 30
# 		self.target_lambda = 0.001
# 		self.convergence = 0.02
# 		self.iteration_limit = 200

# 	"""
# 	def buildAllParts(self):
# 		'''
# 			Build the robot mesh with all parts.
# 			Robot is created in its neutral pose.
# 		'''

# 		# Construct the arm parts
# 		self.Part1 = self.createArmPartMesh(self.L1)
# 		self.Part2 = self.createArmPartMesh(self.L2)
# 		self.Part3 = self.createArmPartMesh(self.L3)
# 		self.Part4 = self.createCoordinateFrameMesh()  # End effector (not an actual frame part)
# 	"""

# 	def setPose(self, Phi, parts):
# 		'''
# 			Set pose of the robot arm.
# 		'''

# 		# Obtain local-to-global matrices from forward kinematics
# 		T_00, T_01, T_02, T_03, T_04, e = self.forward_kinematics(Phi)

# 		# Re-create robot in its neural position
# 		# self.buildAllParts()

# 		# Transforms parts to position it at its correct
# 		# location and orientation.
# 		Base  = parts[0].clone()
# 		Part1 = parts[1].clone()
# 		Part2 = parts[2].clone()
# 		Part3 = parts[3].clone()
# 		Part4 = self.createCoordinateFrameMesh()  # End effector (not an actual frame part)


# 		Base.apply_transform(LinearTransform(T_00))
# 		Part1.apply_transform(LinearTransform(T_01))
# 		Part2.apply_transform(LinearTransform(T_02))
# 		Part3.apply_transform(LinearTransform(T_03))
# 		Part4.apply_transform(LinearTransform(T_04))

# 		return [Base, Part1, Part2, Part3, Part4]


# 	def RotationMatrix(self, theta, axis_name):
# 		""" calculate single rotation of $theta$ matrix around x,y or z
# 		# code from: https://programming-surgeon.com/en/euler-angle-python-en/
# 		input
# 	        theta = rotation angle(degrees)
# 	        axis_name = 'x', 'y' or 'z'
# 	    output
# 	        3x3 rotation matrix
# 	    """

# 		c = np.cos(theta * np.pi / 180)
# 		s = np.sin(theta * np.pi / 180)

# 		if axis_name =='x':
# 			rotation_matrix = np.array([[1, 0,  0],
# 	                                [0, c, -s],
# 	                                [0, s,  c]])
# 		if axis_name =='y':
# 			rotation_matrix = np.array([[ c,  0, s],
# 	                                [ 0,  1, 0],
# 	                                [-s,  0, c]])
# 		elif axis_name =='z':
# 			rotation_matrix = np.array([[c, -s, 0],
# 	                                [s,  c, 0],
# 	                                [0,  0, 1]])
# 		return rotation_matrix

# 	def createCoordinateFrameMesh(self):
# 		"""Returns the mesh representing a coordinate frame
# 	    Args:
# 	      No input args
# 	    Returns:
# 	      F: vedo.mesh object (arrows for axis)
# 		"""
# 		_shaft_radius = 0.05
# 		_head_radius 	= 0.10
# 		_alpha 				= 1
# 		unit 					= 30

# 	    # x-axis as an arrow
# 		x_axisArrow = Arrow(start_pt=(0, 0, 0),
# 	                        end_pt=(unit, 0, 0),
# 	                        s=None,
# 	                        shaft_radius=_shaft_radius,
# 	                        head_radius=_head_radius,
# 	                        head_length=None,
# 	                        res=12,
# 	                        c='red',
# 	                        alpha=_alpha)

# 	    # y-axis as an arrow
# 		y_axisArrow = Arrow(start_pt=(0, 0, 0),
# 	                        end_pt=(0, unit, 0),
# 	                        s=None,
# 	                        shaft_radius=_shaft_radius,
# 	                        head_radius=_head_radius,
# 	                        head_length=None,
# 	                        res=12,
# 	                        c='green',
# 	                        alpha=_alpha)

# 	    # z-axis as an arrow
# 		z_axisArrow = Arrow(start_pt=(0, 0, 0),
# 	                        end_pt=(0, 0, unit),
# 	                        s=None,
# 	                        shaft_radius=_shaft_radius,
# 	                        head_radius=_head_radius,
# 	                        head_length=None,
# 	                        res=12,
# 	                        c='blue',
# 	                        alpha=_alpha)

# 		originDot = Sphere(pos=[0,0,0],
# 	                       c="black",
# 	                       r=0.10*unit)


# 	    # Combine the axes together to form a frame as a single mesh object
# 		F = x_axisArrow + y_axisArrow + z_axisArrow + originDot

# 		return F


# 	def getLocalFrameMatrix(self, R_ij, t_ij):
# 		"""Returns the matrix representing the local frame
# 	    Args:
# 	      R_ij: rotation of Frame j w.r.t. Frame i
# 	      t_ij: translation of Frame j w.r.t. Frame i
# 	    Returns:
# 	      T_ij: Matrix of Frame j w.r.t. Frame i.

# 	    """
# 		# Rigid-body transformation [ R t ]
# 		T_ij = np.block([[R_ij,                t_ij],
# 	                   [np.zeros((1, 3)),       1]])

# 		return T_ij



# 	def forward_kinematics(self, Phi):

# 		# Radius of the sphere representing the joint
# 		radius = 0.4

# 		# Joint angle
# 		phi1 =  Phi[0]    # Rotation angle of part 1 in degrees


# 		# Matrix of Robot Base (written w.r.t. Coordinate origin, which is the previous frame)
# 		R_00 = self.RotationMatrix(0, axis_name = 'z')   	# Rotation matrix, 0 rotation
# 		t_00 = np.copy(self.arm_location)                            # Frame's origin (w.r.t. previous frame)

# 		t_00[-1] = 0
# 		T_00 = self.getLocalFrameMatrix(R_00, t_00)         # Matrix of Robot Base w.r.t. Coordinate origin


# 		# Matrix of Frame 1 (written w.r.t. Frame 0, which is the previous frame)
# 		R_01 = self.RotationMatrix(phi1, axis_name = 'z')   # Rotation matrix
# 		t_01 = self.arm_location                            # Frame's origin (w.r.t. previous frame)
# 		T_01 = self.getLocalFrameMatrix(R_01, t_01)         # Matrix of Frame 1 w.r.t. Frame 0

# 		# Joint angle
# 		phi2 = Phi[1]    # Rotation angle of part 2 in degrees

# 		# Matrix of Frame 2 (written w.r.t. Frame 1, which is the previous frame)
# 		R_12 = self.RotationMatrix(phi2, axis_name = 'y')   # Rotation matrix
# 		t_12 = np.array([[0.0], [0.0], [self.L1+2*radius]])  # Frame's origin (w.r.t. previous frame)
# 		T_12 = self.getLocalFrameMatrix(R_12, t_12)         # Matrix of Frame 2 w.r.t. Frame 1

# 		# Matrix of Frame 2 w.r.t. Frame 0 (i.e., the world frame)
# 		T_02 = T_01 @ T_12


# 		phi3 = Phi[2]    # Rotation angle of the end-effector in degrees

# 		# Matrix of Frame 3 (written w.r.t. Frame 2, which is the previous frame)
# 		R_23 = self.RotationMatrix(phi3, axis_name = 'y')   # Rotation matrix
# 		t_23   = np.array([[0.0], [0.0], [self.L2+2*radius]])  # Frame's origin (w.r.t. previous frame)

# 		# Matrix of Frame 3 w.r.t. Frame 2
# 		T_23 = self.getLocalFrameMatrix(R_23, t_23)

# 		# Matrix of Frame 3 w.r.t. Frame 0 (i.e., the world frame)
# 		T_03 = T_01 @ T_12 @ T_23

# 		phi4 = Phi[3]

# 		# Matrix of Frame 3 (written w.r.t. Frame 2, which is the previous frame)
# 		R_34 = self.RotationMatrix(phi4, axis_name = 'y')   # Rotation matrix
# 		t_34   = np.array([[-28.4], [0.0], [self.L3+radius]])  # Frame's origin (w.r.t. previous frame)

# 		# Matrix of Frame 3 w.r.t. Frame 2
# 		T_34 = self.getLocalFrameMatrix(R_34, t_34)

# 		# Matrix of Frame 3 w.r.t. Frame 0 (i.e., the world frame)
# 		T_04 = T_01 @ T_12 @ T_23 @ T_34

# 		e = T_04[0:3,-1]    # Last column of the last frame matrix

# 		return T_00, T_01, T_02, T_03, T_04, e

# 	def jacobian_matrix(self, phi):

# 		step = np.deg2rad(self.delta_phi)  # radians

# 		# Obtain local-to-global matrices from forward kinematics
# 		_, _, _, _, _, e = self.forward_kinematics(phi)

# 		# phi 1 gradient
# 		_, _, _, _, _, e_phi1_delta = self.forward_kinematics(phi + np.array([step, 0, 0, 0]))
# 		e_phi1_derive = (e_phi1_delta - e) / step

# 		# phi 2 gradient
# 		_, _, _, _, _, e_phi2_delta = self.forward_kinematics(phi + np.array([0, step, 0, 0]))
# 		e_phi2_derive = (e_phi2_delta - e) / step

# 		# phi 3 gradient
# 		_, _, _, _, _, e_phi3_delta = self.forward_kinematics(phi + np.array([0, 0, step, 0]))
# 		e_phi3_derive = (e_phi3_delta - e) / step

# 		# Jacobian columns
# 		jacobian = np.concatenate((e_phi1_derive.reshape((3,1)), e_phi2_derive.reshape((3,1)), e_phi3_derive.reshape((3,1))), axis=1)

# 		# print(e_phi1_derive)
# 		# print(jacobian)
# 		return jacobian

# 	def snapshot(self, recorder, phi, parts):
# 		Part = self.setPose(phi, parts)
# 		recorder.append(Part)

# 	def inverse_kinematics_newton(self, initial_phi, parts):

# 		phi  							= initial_phi
# 		_, _, _, _, _, e 	= self.forward_kinematics(initial_phi)
# 		target 						= self.target
# 		target_lambda 		= self.target_lambda
# 		convergence 			= self.convergence
# 		target_tolerance 	= self.target_tolerance
# 		iteration_limit 	= self.iteration_limit
# 		recorder 					= []
# 		iteration 				= 0
# 		e_accumulate 			= 0

# 		while abs(np.linalg.norm(target - e)) > target_tolerance:
# 			iteration += 1
# 			jacobian_matrix = self.jacobian_matrix(phi)
# 			jacobian_pseudo_inverse = np.linalg.pinv(jacobian_matrix)
# 			e_delta = target_lambda*(target - e)
# 			phi_delta = jacobian_pseudo_inverse @ e_delta
# 			phi_delta = np.append(phi_delta, [0], axis = 0)
# 			phi = phi + phi_delta
# 			e_previous = e
# 			_, _, _, _, _, e = self.forward_kinematics(phi)
# 			e_accumulate += np.linalg.norm(e_previous - e)

# 			# print(abs(np.linalg.norm(target - e)))
# 			# print(abs(np.linalg.norm(e_previous - e)))

# 			# snap shot
# 			if iteration % 20 == 0 or e_accumulate > 10:
# 				self.snapshot(recorder, phi, parts)
# 				e_accumulate = 0

# 			# gradient descent exit criteria 1
# 			if abs(np.linalg.norm(e_previous - e)) < convergence:
# 				break

# 			# gradient descent exit criteria 2
# 			if iteration > iteration_limit:
# 				break

# 		return recorder



In [ ]:
from vedo import load, Axes, Circle, Box, Arrow
import numpy as np

import os
import glob
import numpy as np
import vedo
from vedo import Plotter, Sphere, Plane


"""
Settings
"""
# component heights in mm, need to get aligned with stl models
unit = 1
BaseH = 105/unit
BaseRotH = 81/unit
HumerusH = 217/unit
RadiusH = 416/unit

BaseX = 10/unit
BaseY = 10/unit

Circle_radius = 550/unit

BasePos = [ [Circle_radius, 0],
			[Circle_radius * np.cos(120 * np.pi / 180), Circle_radius * np.sin(120 * np.pi / 180)],
			[Circle_radius * np.cos(-120 * np.pi / 180), Circle_radius * np.sin(-120 * np.pi / 180)]
			]

L = [BaseRotH, HumerusH, RadiusH, 0]
# Animation parameters
view_angle = [1,1,1]
degree1Max = 60
degree2Max = 60
degree3Max = 60

step = 5

# load parts from stl file
robot_dir = "robot/"
Base 			= load(robot_dir + 'Base.stl').color('blue5')
BaseRot 	= load(robot_dir + 'BaseRot.stl').color('lightblue')
Humerus 	= load(robot_dir + 'Humerus.stl').color('gray5')
Radius 		= load(robot_dir + 'Radius.stl').color('red5')

parts 		= [Base, BaseRot, Humerus, Radius]

# arm_location = np.array([[BaseX],[BaseY], [BaseH]])    	# Arm location (position of the first frame)
arm_location0 = np.array([[BasePos[0][0]],[BasePos[0][1]], [BaseH]])    	# Arm location (position of the first frame)
arm_location1 = np.array([[BasePos[1][0]],[BasePos[1][1]], [BaseH]])    	# Arm location (position of the first frame)
arm_location2 = np.array([[BasePos[2][0]],[BasePos[2][1]], [BaseH]])    	# Arm location (position of the first frame)

# create instances
myRobot0 = RobotArm(L, parts, arm_location0)             # The robot arm
myRobot1 = RobotArm(L, parts, arm_location1)             # The robot arm
myRobot2 = RobotArm(L, parts, arm_location2)             # The robot arm

# Set the limits of the graph x, y, and z ranges
axes = Axes(xrange=(-Circle_radius*1.2,Circle_radius*1.2), yrange=(-Circle_radius*1.2,Circle_radius*1.2), zrange=(0,Circle_radius*1.2))
circle = Circle(pos=(0, 0, 0), r=Circle_radius, res=120, c='lightgray', alpha=1.0)
# declare the class instance
plt = Plotter(size=(300, 300), bg="white", axes=10, offscreen=True, interactive=False)
plt.show(axes, viewup = view_angle)

# A short sequence of poses
Poses = np.array([[  0,  0,  0,  0],
					[-30, 50, 30,  0],
					[ 30,-50,-30,  0],
					[  0,  0,  0,  0]])


In [ ]:
snapshot0 = myRobot0.inverse_kinematics_newton(Poses[0], parts)
snapshot1 = myRobot1.inverse_kinematics_newton(Poses[0], parts)
snapshot2 = myRobot2.inverse_kinematics_newton(Poses[0], parts)


In [ ]:
snapshot0

In [ ]:





ball = Sphere(myRobot0.target, r=30).c("red")

idxlimit = max(len(snapshot0), len(snapshot1), len(snapshot2))

# animation
for idx in range(idxlimit):
	# All objects to Plotter.
	# Clear plot before showing new iteration
	plt.clear()
	plt += axes
	plt += circle
	plt += ball
	if idx >= len(snapshot0):
		plt += snapshot0[len(snapshot0)-1]
	else:
		plt += snapshot0[idx]
	if idx >= len(snapshot1):
		plt += snapshot1[len(snapshot1)-1]
	else:
		plt += snapshot1[idx]
	if idx >= len(snapshot2):
		plt += snapshot2[len(snapshot2)-1]
	else:
		plt += snapshot2[idx]

	idx += 10
	# Show scene
	plt.render()    #  What is the difference between render() and show()?
	plt.screenshot(f"{frames_dir}/frame_{idx:03d}.png")




# New version

In [ ]:
#@title Imports
import os
import glob
import numpy as np
import vedo

from vedo import (
    load,
    Axes,
    Circle,
    Arrow,
    Sphere,
    Plotter,
    LinearTransform,
)

In [ ]:
#@title Robot class

class RobotArm:
    """
    Robot arm with:
      - forward kinematics
      - Jacobian-based IK
      - persistent vedo meshes updated in-place
    """

    def __init__(self, partLengths, parts, arm_location):
        # Arm location (position of the first frame)
        self.arm_location = np.array(arm_location, dtype=float)

        # Lengths
        self.L1, self.L2, self.L3, self.L4 = partLengths

        # Store source meshes (do not mutate these directly)
        self.source_parts = parts

        # Constants
        self.delta_phi = 0.1              # finite-difference step, in degrees
        self.target = np.array([0, 100, 200], dtype=float)
        self.target_tolerance = 30
        self.target_lambda = 0.001
        self.convergence = 0.02
        self.iteration_limit = 200

        # Live meshes for rendering
        self.meshes = None
        self.initialize_meshes()

    def initialize_meshes(self):
        """
        Create robot meshes once for this robot instance.
        These are the only meshes used during animation.
        """
        Base  = self.source_parts[0].clone()
        Part1 = self.source_parts[1].clone()
        Part2 = self.source_parts[2].clone()
        Part3 = self.source_parts[3].clone()
        Part4 = self.createCoordinateFrameMesh()

        self.meshes = [Base, Part1, Part2, Part3, Part4]

    def RotationMatrix(self, theta, axis_name):
        """
        Single-axis rotation matrix.
        theta is interpreted in degrees.
        """
        c = np.cos(theta * np.pi / 180.0)
        s = np.sin(theta * np.pi / 180.0)

        if axis_name == "x":
            rotation_matrix = np.array([
                [1, 0,  0],
                [0, c, -s],
                [0, s,  c]
            ])
        elif axis_name == "y":
            rotation_matrix = np.array([
                [ c, 0, s],
                [ 0, 1, 0],
                [-s, 0, c]
            ])
        elif axis_name == "z":
            rotation_matrix = np.array([
                [c, -s, 0],
                [s,  c, 0],
                [0,  0, 1]
            ])
        else:
            raise ValueError(f"Unknown axis_name: {axis_name}")

        return rotation_matrix

    def createCoordinateFrameMesh(self):
        """
        Returns a mesh representing a coordinate frame.
        """
        shaft_radius = 0.05
        head_radius = 0.10
        alpha = 1.0
        unit = 30

        x_axisArrow = Arrow(
            start_pt=(0, 0, 0),
            end_pt=(unit, 0, 0),
            shaft_radius=shaft_radius,
            head_radius=head_radius,
            res=12,
            c="red",
            alpha=alpha,
        )

        y_axisArrow = Arrow(
            start_pt=(0, 0, 0),
            end_pt=(0, unit, 0),
            shaft_radius=shaft_radius,
            head_radius=head_radius,
            res=12,
            c="green",
            alpha=alpha,
        )

        z_axisArrow = Arrow(
            start_pt=(0, 0, 0),
            end_pt=(0, 0, unit),
            shaft_radius=shaft_radius,
            head_radius=head_radius,
            res=12,
            c="blue",
            alpha=alpha,
        )

        originDot = Sphere(pos=[0, 0, 0], c="black", r=0.10 * unit)

        F = x_axisArrow + y_axisArrow + z_axisArrow + originDot
        return F

    def getLocalFrameMatrix(self, R_ij, t_ij):
        """
        Homogeneous transform matrix T_ij from rotation R_ij and translation t_ij.
        """
        T_ij = np.block([
            [R_ij, t_ij],
            [np.zeros((1, 3)), np.array([[1]])]
        ])
        return T_ij

    def forward_kinematics(self, Phi):
        """
        Returns:
            T_00, T_01, T_02, T_03, T_04, e
        where e is the end-effector position.
        """
        radius = 0.4

        phi1 = Phi[0]

        # Base
        R_00 = self.RotationMatrix(0, axis_name="z")
        t_00 = np.copy(self.arm_location)
        t_00[-1] = 0
        T_00 = self.getLocalFrameMatrix(R_00, t_00)

        # Frame 1
        R_01 = self.RotationMatrix(phi1, axis_name="z")
        t_01 = self.arm_location
        T_01 = self.getLocalFrameMatrix(R_01, t_01)

        # Frame 2
        phi2 = Phi[1]
        R_12 = self.RotationMatrix(phi2, axis_name="y")
        t_12 = np.array([[0.0], [0.0], [self.L1 + 2 * radius]])
        T_12 = self.getLocalFrameMatrix(R_12, t_12)
        T_02 = T_01 @ T_12

        # Frame 3
        phi3 = Phi[2]
        R_23 = self.RotationMatrix(phi3, axis_name="y")
        t_23 = np.array([[0.0], [0.0], [self.L2 + 2 * radius]])
        T_23 = self.getLocalFrameMatrix(R_23, t_23)
        T_03 = T_01 @ T_12 @ T_23

        # Frame 4 / end effector
        phi4 = Phi[3]
        R_34 = self.RotationMatrix(phi4, axis_name="y")
        t_34 = np.array([[-28.4], [0.0], [self.L3 + radius]])
        T_34 = self.getLocalFrameMatrix(R_34, t_34)
        T_04 = T_01 @ T_12 @ T_23 @ T_34

        e = T_04[0:3, -1]
        return T_00, T_01, T_02, T_03, T_04, e

    def get_pose_transforms(self, Phi):
        """
        Convenience function for rendering.
        """
        T_00, T_01, T_02, T_03, T_04, _ = self.forward_kinematics(Phi)
        return [T_00, T_01, T_02, T_03, T_04]

    def update_pose(self, Phi):
        """
        Update the robot's existing meshes in-place.
        No cloning, no mesh accumulation.
        """
        transforms = self.get_pose_transforms(Phi)

        # Re-clone from the original neutral meshes each frame to avoid
        # transform accumulation issues across frames.
        Base  = self.source_parts[0].clone()
        Part1 = self.source_parts[1].clone()
        Part2 = self.source_parts[2].clone()
        Part3 = self.source_parts[3].clone()
        Part4 = self.createCoordinateFrameMesh()

        new_meshes = [Base, Part1, Part2, Part3, Part4]

        for mesh, T in zip(new_meshes, transforms):
            mesh.apply_transform(LinearTransform(T))

        self.meshes = new_meshes
        return self.meshes

    def jacobian_matrix(self, phi):
        """
        Numerical Jacobian of end-effector position wrt joint angles.
        """
        # Keep everything in degrees to match the rest of your code
        step = self.delta_phi

        _, _, _, _, _, e = self.forward_kinematics(phi)

        _, _, _, _, _, e_phi1_delta = self.forward_kinematics(phi + np.array([step, 0, 0, 0]))
        e_phi1_derive = (e_phi1_delta - e) / step

        _, _, _, _, _, e_phi2_delta = self.forward_kinematics(phi + np.array([0, step, 0, 0]))
        e_phi2_derive = (e_phi2_delta - e) / step

        _, _, _, _, _, e_phi3_delta = self.forward_kinematics(phi + np.array([0, 0, step, 0]))
        e_phi3_derive = (e_phi3_delta - e) / step

        jacobian = np.concatenate(
            (
                e_phi1_derive.reshape((3, 1)),
                e_phi2_derive.reshape((3, 1)),
                e_phi3_derive.reshape((3, 1)),
            ),
            axis=1,
        )

        return jacobian

    def inverse_kinematics_newton(self, initial_phi, record_every=20, motion_threshold=10):
        """
        Runs IK and stores only joint angles.

        Returns
        -------
        trajectory : (N, 4) ndarray
            Recorded joint-angle states.
        """
        phi = np.array(initial_phi, dtype=float).copy()
        _, _, _, _, _, e = self.forward_kinematics(phi)

        target = np.array(self.target, dtype=float)
        target_lambda = self.target_lambda
        convergence = self.convergence
        target_tolerance = self.target_tolerance
        iteration_limit = self.iteration_limit

        recorder = [phi.copy()]
        iteration = 0
        e_accumulate = 0.0

        while np.linalg.norm(target - e) > target_tolerance:
            iteration += 1

            J = self.jacobian_matrix(phi)
            J_pinv = np.linalg.pinv(J)

            e_delta = target_lambda * (target - e)
            phi_delta = J_pinv @ e_delta
            phi_delta = np.append(phi_delta, [0.0])

            phi = phi + phi_delta

            e_previous = e
            _, _, _, _, _, e = self.forward_kinematics(phi)
            e_accumulate += np.linalg.norm(e_previous - e)

            if iteration % record_every == 0 or e_accumulate > motion_threshold:
                recorder.append(phi.copy())
                e_accumulate = 0.0

            if np.linalg.norm(e_previous - e) < convergence:
                break

            if iteration > iteration_limit:
                break

        # Always include final pose
        if not np.allclose(recorder[-1], phi):
            recorder.append(phi.copy())

        return np.array(recorder)

In [ ]:
# -----------------------------
#@title Clear video frames
# -----------------------------
frames_dir = "frames"
os.makedirs(frames_dir, exist_ok=True)

for f in glob.glob(os.path.join(frames_dir, "*.png")):
    os.remove(f)

print("Old frames cleared.")

In [ ]:
#@title Scene setup and robot creation
#@markdown This cell loads the STL files, creates the three robots, and prepares the common scene.

# --------------------------------------------------
# Settings
# --------------------------------------------------
unit = 1
BaseH = 105 / unit
BaseRotH = 81 / unit
HumerusH = 217 / unit
RadiusH = 416 / unit

BaseX = 10 / unit
BaseY = 10 / unit

Circle_radius = 550 / unit

BasePos = [
    [Circle_radius, 0],
    [Circle_radius * np.cos(120 * np.pi / 180),  Circle_radius * np.sin(120 * np.pi / 180)],
    [Circle_radius * np.cos(-120 * np.pi / 180), Circle_radius * np.sin(-120 * np.pi / 180)],
]

L = [BaseRotH, HumerusH, RadiusH, 0]

view_angle = [1, 1, 1]

# --------------------------------------------------
# Load meshes
# --------------------------------------------------
robot_dir = "robot/"
Base    = load(robot_dir + "Base.stl").color("blue5")
BaseRot = load(robot_dir + "BaseRot.stl").color("lightblue")
Humerus = load(robot_dir + "Humerus.stl").color("gray5")
Radius  = load(robot_dir + "Radius.stl").color("red5")

parts = [Base, BaseRot, Humerus, Radius]

# --------------------------------------------------
# Arm locations
# --------------------------------------------------
arm_location0 = np.array([[BasePos[0][0]], [BasePos[0][1]], [BaseH]], dtype=float)
arm_location1 = np.array([[BasePos[1][0]], [BasePos[1][1]], [BaseH]], dtype=float)
arm_location2 = np.array([[BasePos[2][0]], [BasePos[2][1]], [BaseH]], dtype=float)

# --------------------------------------------------
# Create robot instances
# --------------------------------------------------
myRobot0 = RobotArm(L, parts, arm_location0)
myRobot1 = RobotArm(L, parts, arm_location1)
myRobot2 = RobotArm(L, parts, arm_location2)

# --------------------------------------------------
# Scene actors
# --------------------------------------------------
axes = Axes(
    xrange=(-Circle_radius * 1.2, Circle_radius * 1.2),
    yrange=(-Circle_radius * 1.2, Circle_radius * 1.2),
    zrange=(0, Circle_radius * 1.2),
)

circle = Circle(pos=(0, 0, 0), r=Circle_radius, res=120, c="lightgray", alpha=1.0)
ball = Sphere(myRobot0.target, r=30).c("red")

# --------------------------------------------------
# Plotter
# --------------------------------------------------
# plt = Plotter(size=(600, 600), bg="white", offscreen=True, interactive=False)

In [ ]:
# Set the limits of the graph x, y, and z ranges
#axes = Axes(xrange=(-Circle_radius*1.2,Circle_radius*1.2), yrange=(-Circle_radius*1.2,Circle_radius*1.2), zrange=(0,Circle_radius*1.2))
#circle = Circle(pos=(0, 0, 0), r=Circle_radius, res=120, c='lightgray', alpha=1.0)
# declare the class instance
plt = Plotter(size=(300, 300), bg="white", axes=10, offscreen=True, interactive=False)
plt.show(axes, viewup = view_angle)

# # A short sequence of poses
# Poses = np.array([[  0,  0,  0,  0],
# 					[-30, 50, 30,  0],
# 					[ 30,-50,-30,  0],
# 					[  0,  0,  0,  0]])


In [ ]:
#@title Define initial joint states and solve IK


# Initial poses for each arm
phi0_init = np.array([0.0, 0.0, 0.0, 0.0])
phi1_init = np.array([0.0, 0.0, 0.0, 0.0])
phi2_init = np.array([0.0, 0.0, 0.0, 0.0])

# Solve IK and store only joint angles
traj0 = myRobot0.inverse_kinematics_newton(phi0_init)
traj1 = myRobot1.inverse_kinematics_newton(phi1_init)
traj2 = myRobot2.inverse_kinematics_newton(phi2_init)

print("traj0 shape:", traj0.shape)
print("traj1 shape:", traj1.shape)
print("traj2 shape:", traj2.shape)



In [ ]:
traj0

In [ ]:
#@title (Optional) Set different targets for each robot

# Example:
# myRobot0.target = np.array([0, 100, 200], dtype=float)
# myRobot1.target = np.array([50, 80, 180], dtype=float)
# myRobot2.target = np.array([-50, 120, 220], dtype=float)



In [ ]:
#@title (Optional) Save trajectories to disk

#@markdown  It can be loaded later without re-running IK:


np.save("traj0.npy", traj0)
np.save("traj1.npy", traj1)
np.save("traj2.npy", traj2)

In [ ]:
#@title (Optional) Load trajectories to disk


traj0 = np.load("traj0.npy")
traj1 = np.load("traj1.npy")
traj2 = np.load("traj2.npy")

In [ ]:
#@title Render animation frames from stored joint angles

#@markdown Reconstructs robot poses from the stored $\phi$ values.

# --------------------------------------------------
# Output folder
# --------------------------------------------------
frames_dir = "frames"
os.makedirs(frames_dir, exist_ok=True)

for f in glob.glob(os.path.join(frames_dir, "*.png")):
    os.remove(f)

# --------------------------------------------------
# Animation
# --------------------------------------------------
idxlimit = max(len(traj0), len(traj1), len(traj2))

for idx in range(idxlimit):
    phi0 = traj0[min(idx, len(traj0) - 1)]
    phi1 = traj1[min(idx, len(traj1) - 1)]
    phi2 = traj2[min(idx, len(traj2) - 1)]

    meshes0 = myRobot0.update_pose(phi0)
    meshes1 = myRobot1.update_pose(phi1)
    meshes2 = myRobot2.update_pose(phi2)

    plt.clear()
    plt += axes
    plt += circle
    plt += ball

    for actor in meshes0:
        plt += actor
    for actor in meshes1:
        plt += actor
    for actor in meshes2:
        plt += actor

    plt.render()
    plt.screenshot(f"{frames_dir}/frame_{idx:03d}.png")

In [ ]:
#@title Show animation sequence as a grid

import glob
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------
# SETTINGS
# -----------------------------
GRID_ROWS = 4
GRID_COLS = 6

# -----------------------------
# Load frames
# -----------------------------
frame_files = sorted(glob.glob(f"{frames_dir}/frame_*.png"))
N = len(frame_files)

grid_size = GRID_ROWS * GRID_COLS

if N == 0:
    raise RuntimeError("No frames found.")

# -----------------------------
# Compute sampling indices
# -----------------------------
if N <= grid_size:
    indices = np.arange(N)
else:
    step = N / grid_size
    indices = (np.arange(grid_size) * step).astype(int)

# -----------------------------
# Display grid preview
# -----------------------------
print("Displaying frame grid preview (sampled across sequence)...")

fig, axes = plt.subplots(GRID_ROWS, GRID_COLS, figsize=(12, 8))

for i in range(grid_size):
    ax = axes.flat[i]
    ax.axis("off")

    if i < len(indices):
        img = Image.open(frame_files[indices[i]])
        ax.imshow(img)

plt.tight_layout()
plt.show()

In [ ]:
#@title Create video from frames
# -----------------------------
# Create video
# -----------------------------
!ffmpeg -loglevel error -y -framerate 30 -i frames/frame_%03d.png \
        -c:v libx264 -pix_fmt yuv420p -movflags +faststart spheres.mp4

print("Video created: spheres.mp4")

In [ ]:
#@title Display video in notebook


from IPython.display import HTML
from base64 import b64encode

video_path = 'spheres.mp4'
# Read the file and encode it
mp4 = open(video_path, 'rb').read()
decoded_vid = "data:video/mp4;base64," + b64encode(mp4).decode()

# Embed in an HTML video tag
HTML(f'<video width=500 controls><source src={decoded_vid} type="video/mp4"></video>')